# Multi-Objective

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
import plotly.io as pio
import plotly.graph_objs as go
from plotly.subplots import make_subplots


def line(error_y_mode=None, **kwargs):
    """Extension of `plotly.express.line` to use error bands."""
    error_modes = {"bar", "band", "bars", "bands", None}

    if error_y_mode not in error_modes:
        raise ValueError(
            f"'error_y_mode' must be one of {error_modes}, received"
            + f" {repr(error_y_mode)}."
        )

    if error_y_mode in {"bar", "bars", None}:
        fig = px.line(**kwargs)
    elif error_y_mode in {"band", "bands"}:
        if "error_y" not in kwargs:
            raise ValueError(
                "If you provide 'error_y_mode' you must also provide 'error_y'."
            )

        kwargs_per_line = [kwargs]

        if (
            kwargs["data_frame"] is not None
            and kwargs["y"] is not None
            and not isinstance(kwargs["y"], str)
        ):
            kwargs_per_line = []

            for i in range(len(kwargs["y"])):
                line_kwargs = kwargs.copy()
                line_kwargs["y"] = line_kwargs["y"][i]
                line_kwargs["error_y"] = line_kwargs["error_y"][i]
                line_kwargs["error_y_minus"] = line_kwargs["error_y_minus"][i]
                kwargs_per_line.append(line_kwargs)

        fig = px.line(
            **{
                arg: val
                for arg, val in kwargs.items()
                if arg not in ("error_y", "error_y_minus")
            }
        )

        for i, line_kwargs in enumerate(kwargs_per_line):
            data = px.line(**line_kwargs).data[0]
            x = list(data["x"])
            y_upper = list(data["y"] + data["error_y"]["array"])
            y_lower = list(
                data["y"] - data["error_y"]["array"]
                if data["error_y"]["arrayminus"] is None
                else data["y"] - data["error_y"]["arrayminus"]
            )

            colors = fig.data[i]["line"]["color"]
            colors = tuple(int(colors.lstrip("#")[j : j + 2], 16) for j in (0, 2, 4))
            color_str = f"rgba({colors},.3)"
            color_str = color_str.replace("((", "(").replace("),", ",").replace(" ", "")

            fig.add_trace(
                go.Scatter(
                    x=x + x[::-1],
                    y=y_upper + y_lower[::-1],
                    fill="toself",
                    fillcolor=color_str,
                    line=dict(color="rgba(255,255,255,0)"),
                    hoverinfo="skip",
                    showlegend=False,
                    legendgroup=data["legendgroup"],
                    xaxis=data["xaxis"],
                    yaxis=data["yaxis"],
                )
            )

        # Reorder data as said here: https://stackoverflow.com/a/66854398/8849755
        reordered_data = []

        for i in range(int(len(fig.data) / 2)):
            reordered_data.append(fig.data[i + int(len(fig.data) / 2)])
            reordered_data.append(fig.data[i])

        fig.data = tuple(reordered_data)

    return fig


pio.templates.default = "plotly_white"
default_layout = dict(
    font_family="FiraMono Nerd Font",
    font_color="#15244c",
    font_size=34,
    legend=dict(
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(0,0,0,0)"
    ),
    margin=dict(l=1, t=1, b=1, r=1)
)

default_xaxes = dict(
    showline=True,
    linecolor="#aeb6c2",
    linewidth=2,
    row=1,
    col=1,
    mirror=True,
    gridcolor="#e6ebeb",
)

default_yaxes = dict(
    showline=True,
    linecolor="#aeb6c2",
    linewidth=2,
    row=1,
    col=1,
    mirror=True,
    gridcolor="#e6ebeb",
    ticksuffix=" ",
    rangemode="tozero",
)

## Load Runs

In [ ]:
from collections.abc import MutableMapping
import pathlib as pl
import time

import json
import numpy as np
import wandb


def load_runs(folder_path):
    run_files = pl.Path(folder_path).glob("*.json")
    task = folder_path.rstrip("/").rsplit("/", 1)[-1]
    runs = {}

    for file in run_files:
        with open(file, "r") as file:
            data = json.load(file)
            
        seed = int(data["config"]["arch"]["seed"])
        expansion_steps =  int(data["config"]["system"]["expansion_steps"])
        beta = float(data["config"]["system"]["beta"])
        gamma = float(data["config"]["system"]["gamma"])
        h = int(data["config"]["system"]["h"])
        die_prob = float(data["config"]["env"]["kwargs"]["die_prob"])
        rnd_prob = float(data["config"]["env"]["kwargs"]["rnd_move_prob"])
        n_enemies = len(eval(data["config"]["env"]["kwargs"]["enemies_pos"]))
        cost_fn = data["config"]["env"]["kwargs"]["cost"]
        expl_type = data["config"]["system"]["bonus"]["bonus_type"]
        expl_coeff = None
        expl_coeff = float(data["config"]["system"]["bonus"]["kw"]["coeff"])
        expl_kw = f"-coef={expl_coeff}"
        tag = f"occ/gamma={gamma}/h={h}/beta={beta}/expl={expl_type}{expl_kw}/expansion={expansion_steps}"
        tag += f"/die_prob={die_prob}/rnd_prob={rnd_prob}/n_enemies={n_enemies}/cost={cost_fn}"

        if tag not in runs:
            runs[tag] = {"seed": [], "costs": [], "erm": [], "states": [], "actions": []}
            runs[tag]["beta"] = beta
            runs[tag]["expl"] = expl_type
            runs[tag]["expl_coeff"] = expl_coeff
            runs[tag]["expansion"] = expansion_steps

        cost = data["TRAIN"]["cost"][0]
        erm = data["TRAIN"]["erm"]
        states = data["TRAIN"]["state"]
        actions = data["TRAIN"]["action"]
        assert isinstance(cost, float)
        assert len(erm) == len(actions) == h
        assert len(states) == len(actions) + 1
        assert all([(e is not None) for e in erm])
        assert all([(s is not None) for s in states])
        assert all([(a is not None) for a in actions])
        assert isinstance(cost, float)
        runs[tag]["seed"].append(seed)
        runs[tag]["costs"].append(cost)
        runs[tag]["erm"].append(erm)
        runs[tag]["states"].append(states)
        runs[tag]["actions"].append(actions)
        
    for tag in runs:
        runs[tag]["seed"] = np.array(runs[tag]["seed"])
        runs[tag]["costs"] = np.array(runs[tag]["costs"])
        runs[tag]["erm"] = np.stack(runs[tag]["erm"], axis=0)
        runs[tag]["states"] = np.stack(runs[tag]["states"], axis=0)
        runs[tag]["actions"] = np.stack(runs[tag]["actions"], axis=0)

    return runs
    
def _flatten(dictionary, parent_key='', separator='/'):
    items = []

    for key, value in dictionary.items():
        new_key = parent_key + separator + key if parent_key else key
        
        if isinstance(value, MutableMapping):
            items.extend(_flatten(value, new_key, separator=separator).items())
        else:
            items.append((new_key, value))

    return dict(items)

data_keys = ["seed", "costs", "erm", "states", "actions"]
runs = load_runs("../results/erm_occupancy_mcts/40/0.99/mo")  # NOTE: Change this to match with the hyper-params used

# Check that there are no repeated seed
all_seeds = [runs[r]["seed"] for r in runs]
all_seeds = np.concatenate(all_seeds).tolist()
assert len(all_seeds) == len(set(all_seeds)), "Repeated seeds"

data = _flatten({t: {k: runs[t][k] for k in data_keys} for t in runs})

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


df = {
    "erm": np.array([]),
    "gamma": [],
    "h": [],
    "beta": [],
    "policy": [],
    "bonus": [],
    "die_prob": [],
    "rnd_prob": [],
    "n_enemies": [], 
    "cost_fn": []
}

for k, arr in data.items():
    if "/costs" not in k:
        continue

    gamma = float(k.split("/")[1].split("=", 1)[1])
    h = float(k.split("/")[2].split("=", 1)[1])
    beta = float(k.split("/")[3].split("=", 1)[1])
    bonus = k.split("/")[4].split("-", 1)[0].split("=", 1)[1]
    expansion = int(k.split("/")[5].split("=", 1)[1])
    die_prob = float(k.split("/")[6].split("=", 1)[1])
    rnd_prob = float(k.split("/")[7].split("=", 1)[1])
    n_enemies = float(k.split("/")[8].split("=", 1)[1])
    cost_fn = k.split("/")[9].split("=", 1)[1]
    df["erm"] = np.concatenate((df["erm"], arr), axis=0)
    df["gamma"] += [gamma] * arr.shape[0]
    df["h"] += [h] * arr.shape[0]
    df["beta"] += [beta] * arr.shape[0]
    df["policy"] += [f"MCTS ({expansion})"] * arr.shape[0]
    df["bonus"] += [bonus] * arr.shape[0]
    df["die_prob"] += [die_prob] * arr.shape[0]
    df["rnd_prob"] += [rnd_prob] * arr.shape[0]
    df["n_enemies"] += [n_enemies] * arr.shape[0]
    df["cost_fn"] += [cost_fn] * arr.shape[0]

df = pd.DataFrame(df)
df["beta"] = df["beta"].round(3)
betas = [str(beta) for beta in np.sort(df["beta"].unique())]
policies = set(df["policy"].unique())

## Plots

In [ ]:
_df = df.copy()

# NOTE: Change this to match with the hyper-params used
_df = _df[(_df["gamma"] == 0.99) & (_df["h"] == 40) & (_df["cost_fn"] == "root-gold")]
_df = _df[_df["bonus"] == "erm_uct"]
_df = _df[(_df["die_prob"] == 0.025) & (_df["rnd_prob"] == 0.05) & (_df["n_enemies"] == 2)]
_df["beta"] = _df["beta"].astype(str)
betas = [str(beta) for beta in np.sort(_df["beta"].unique())]
fig = px.box(_df, x="beta", y="erm", category_orders={"beta": betas})

_default_layout = default_layout | dict(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=0.99,
    bgcolor="rgba(0,0,0,0)"
))

fig.update_traces(boxmean=True)
fig.update_layout(legend_title_text="algorithm", boxmode="group", **_default_layout)
fig.update_xaxes(title="beta", **default_xaxes)
time.sleep(2)
fig.write_image("imgs/mo_cost_dist.pdf")
time.sleep(2)
fig.show()
     

In [ ]:
import numpy as np
import yaml


# define hyper-params to filter runs
# NOTE: Change this to match with the hyper-params used
hyper_params = {"gamma": 0.99, "bonus": "erm_uct", "die_prob": 0.025, "rnd_prob": 0.05, "n_enemies": 2}


# get difficult terrain positions - `dt_pos`
with open('../src/risk_aware_gumdp/configs/env/mo.yaml', 'r') as file:
    cfg = yaml.safe_load(file)

grid_size = cfg["kwargs"]["size"]
enemies_pos = cfg["kwargs"]["enemies_pos"]
enemies_pos = [x * grid_size[1] + y for (x, y) in enemies_pos]
enemies_pos = np.array(enemies_pos)
gold_pos = [cfg["kwargs"]["gold_pos"][0][0] * grid_size[1] + cfg["kwargs"]["gold_pos"][0][1]]
diamond_pos = [cfg["kwargs"]["diamond_pos"][0][0] * grid_size[1] + cfg["kwargs"]["diamond_pos"][0][1]]
die_idx = 1 + 4 * grid_size[0] * grid_size[1]
got_none = die_idx + 1
got_diamond = got_none + 1
got_gold = got_diamond + 1
got_both = got_gold + 1

# for each run count number of times the agent visited difficult terrain
count = {}

for tag in data:
    gamma = float(tag.split("/")[1].split("=", 1)[1])
    bonus = tag.split("/")[4].split("-", 1)[0].split("=", 1)[1]
    die_prob = float(tag.split("/")[6].split("=", 1)[1])
    rnd_prob = float(tag.split("/")[7].split("=", 1)[1])
    n_enemies = int(tag.split("/")[8].split("=", 1)[1])
    data_type = tag.split("/")[-1]
    cond = gamma == hyper_params["gamma"]
    cond = cond and bonus == hyper_params["bonus"]
    cond = cond and die_prob == hyper_params["die_prob"]
    cond = cond and rnd_prob == hyper_params["rnd_prob"]
    cond = cond and n_enemies == hyper_params["n_enemies"]
    cond = cond and data_type == "states"
    
    if not cond:
        continue

    states = data[tag]  # this is a `np.ndarray`
    count[tag] = {"defeated": 0, "none": 0, "diamond": 0, "gold": 0, "both": 0}
    total = 0
    
    for ep in states:
        total += 1

        if np.any(np.isin(got_none, ep)):
            count[tag]["none"] += 1
            assert not np.all(np.isin(got_diamond, ep))
            assert not np.all(np.isin(got_gold, ep))
            assert not np.all(np.isin(got_both, ep))
        elif np.any(np.isin(got_diamond, ep)):
            count[tag]["diamond"] += 1
            assert not np.all(np.isin(got_none, ep))
            assert not np.all(np.isin(got_gold, ep))
            assert not np.all(np.isin(got_both, ep))
        elif np.any(np.isin(got_gold, ep)):
            count[tag]["gold"] += 1
            assert not np.all(np.isin(got_none, ep))
            assert not np.all(np.isin(got_diamond, ep))
            assert not np.all(np.isin(got_both, ep))
        elif np.any(np.isin(got_both, ep)):
            count[tag]["both"] += 1
            assert not np.all(np.isin(got_none, ep))
            assert not np.all(np.isin(got_diamond, ep))
            assert not np.all(np.isin(got_gold, ep))
        else:
            count[tag]["defeated"] += 1
            assert not np.all(np.isin(got_none, ep))
            assert not np.all(np.isin(got_diamond, ep))
            assert not np.all(np.isin(got_gold, ep))
            assert not np.all(np.isin(got_both, ep))
            
# create custom dataframe with visit information
count_df = {"count": np.array([]), "retrieved": [], "beta": [], "policy": []}

for tag in count:
    beta = float(tag.split("/")[3].split("=", 1)[1])
    expansion = int(tag.split("/")[5].split("=", 1)[1])
    
    for k in count[tag]:
        if k == "gold":
            k_tag = "resource 1"
        elif k == "diamond":
            k_tag = "resource 2"
        elif k == "both":
            k_tag = "both resources"
        elif k == "none":
            k_tag = "no resource"
        else:
            k_tag = k
            
        n_times = np.ones(count[tag][k], dtype=np.int32)
        count_df["count"] = np.concatenate((count_df["count"], n_times / total), axis=0)
        count_df["retrieved"] += [k_tag] * n_times.shape[0]
        count_df["beta"] += [beta] * n_times.shape[0]
        count_df["policy"] += [f"MCTS ({expansion})"] * n_times.shape[0]

count_df = pd.DataFrame(count_df)
count_df["beta"] = count_df["beta"].round(3)
count_df["beta"] = count_df["beta"].astype(str)
retrieved = ["defeated", "no resource", "resource 1", "resource 2", "both resources"]

# plot visit count data
fig = px.histogram(
    count_df,
    x="beta",
    y="count",
    color="retrieved",
    barmode="group",
    category_orders={"retrieved": retrieved, "beta": betas}
)

_default_layout = default_layout | dict(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=0.99,
    bgcolor="rgba(0,0,0,0)"
))


fig.update_layout(legend_title_text="Outcome", boxmode="group", **_default_layout)
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.02,
    xanchor="right",
    x=1
))
fig.update_xaxes(title="beta", **default_xaxes)
fig.update_yaxes(title="Visit Count", **default_yaxes)
time.sleep(2)
fig.write_image("imgs/mo_gold_retrieved_dist.pdf")
time.sleep(2)
fig.show()